- The Scenario: You act as a university data analyst. The legacy admissions portal crashed during a system migration, and the backup applicant data is a mess. You must clean the "Applicant Records" dataset.
- Handling Missing Data: The dataset is riddled with NaNs. You must use .dropna() to remove rows completely missing an Application ID. You will use .fillna() to replace missing Intended Major and High School State values with "Unknown" that were lost during the export.

In [30]:
import pandas as pd
df = pd.read_csv('admissions_messy.csv')
df

#handling missing data
drop_id= df.dropna(subset=['Application ID'])
drop_id
filled_na = df.fillna('Unknown')
filled_na

,Application ID,Applicant Name,Email,Intended Major,High School State,Application Fee,Submission Date
0,1352.0,wanda DOE,wdoe@gmail.com,Math,NV,$297.91,pending
1,1689.0,PETER JONES,pjones@edu.org,Journalism,Unknown,"$1,939.68",08/14/2023
2,1485.0,wanda MAXIMOFF,wmaximoff@yahoo.com,Math,Unknown,$277.90,2023/06/06
3,1388.0,PETER Parker,pparker@yahoo.com,Agriculture,AZ,$370.25,2023-02-24
4,1031.0,bruce rogers,brogers@stark.com,Physics,NV,$53.93,"Oct 16, 2023"
...,...,...,...,...,...,...,...
1045,1330.0,diana MAXIMOFF,dmaximoff@edu.org,Unknown,AZ,$862.30,10-18-2023
1046,1466.0,STEVE Allen,sallen@edu.org,Journalism,Unknown,$389.22,10-20-2023
1047,1121.0,Charlie ROGERS,crogers@stark.com,History,AZ,"$1,746.27","Aug 17, 2023"
1048,1299.0,peter Wayne,pwayne@dailybugle.com,Journalism,KS,$689.05,2023-06-06


 - Handling Duplicates: A server glitch caused applicants to accidentally submit identical applications twice. You must use .duplicated() to find identical application rows and .drop_duplicates(keep='first') to remove the errors and keep only the original submission.

In [31]:
# Handling Duplicate
duplicate = df[df.duplicated()]
duplicate
drop_duplicate = df.drop_duplicates(keep='first')
drop_duplicate

,Application ID,Applicant Name,Email,Intended Major,High School State,Application Fee,Submission Date
0,1352.0,wanda DOE,wdoe@gmail.com,Math,NV,$297.91,pending
1,1689.0,PETER JONES,pjones@edu.org,Journalism,NaN,"$1,939.68",08/14/2023
2,1485.0,wanda MAXIMOFF,wmaximoff@yahoo.com,Math,NaN,$277.90,2023/06/06
3,1388.0,PETER Parker,pparker@yahoo.com,Agriculture,AZ,$370.25,2023-02-24
4,1031.0,bruce rogers,brogers@stark.com,Physics,NV,$53.93,"Oct 16, 2023"
...,...,...,...,...,...,...,...
1044,1087.0,John Stone,jstone@gmail.com,Computer Science,KS,"$1,582.76",2023/05/27
1045,1330.0,diana MAXIMOFF,dmaximoff@edu.org,NaN,AZ,$862.30,10-18-2023
1046,1466.0,STEVE Allen,sallen@edu.org,Journalism,NaN,$389.22,10-20-2023
1047,1121.0,Charlie ROGERS,crogers@stark.com,History,AZ,"$1,746.27","Aug 17, 2023"


- Data Type Conversions: 
- Application Fee is imported as a string (e.g., "$150.50"). You must remove the symbols and use pd.to_numeric() to turn it into a float.

In [18]:
df['Application Fee'] = df['Application Fee'].astype(str)
df['Application Fee'] = df['Application Fee'].str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['Application Fee'] = pd.to_numeric(df['Application Fee'], errors='coerce')
df['Application Fee']

0        297.91
1       1939.68
2        277.90
3        370.25
4         53.93
         ...   
1045     862.30
1046     389.22
1047    1746.27
1048     689.05
1049    1875.22
Name: Application Fee, Length: 1050, dtype: float64

- Replace “pending” and “unknown” text entries in the dates with the previous valid datetime (forward filling).
- Convert Submission Date from messy text into a proper datetime object using pd.to_datetime().

In [19]:
df['Submission Date'] = df['Submission Date'].replace(['pending','unknown'], pd.NA)
filled_submission_date = df['Submission Date'].ffill()
print(filled_submission_date)

datetime_submission_date = pd.to_datetime(filled_submission_date, format='mixed')
print(datetime_submission_date)

0               <NA>
1         08/14/2023
2         2023/06/06
3         2023-02-24
4       Oct 16, 2023
            ...     
1045      10-18-2023
1046      10-20-2023
1047    Aug 17, 2023
1048      2023-06-06
1049      2023-06-06
Name: Submission Date, Length: 1050, dtype: object
0             NaT
1      2023-08-14
2      2023-06-06
3      2023-02-24
4      2023-10-16
          ...    
1045   2023-10-18
1046   2023-10-20
1047   2023-08-17
1048   2023-06-06
1049   2023-06-06
Name: Submission Date, Length: 1050, dtype: datetime64[ns]


- Convert the Submission Date format from various mixed formats into a uniform "M/D/Y" format.

In [20]:
df['M/D/Y'] = datetime_submission_date.dt.strftime("%m/%d/%Y")
print(df['M/D/Y'])

0              NaN
1       08/14/2023
2       06/06/2023
3       02/24/2023
4       10/16/2023
           ...    
1045    10/18/2023
1046    10/20/2023
1047    08/17/2023
1048    06/06/2023
1049    06/06/2023
Name: M/D/Y, Length: 1050, dtype: object


- String Manipulation: The Applicant Name column has chaotic capitalization and extra spaces. You will use .str.title() and .str.strip() to fix this. You must also extract the email domain (e.g., extracting "yahoo.com" from "student@yahoo.com") using .str.split() to see which email providers are most common.

In [28]:
df['Applicant Name'] = df['Applicant Name'].str.title()
df['Applicant Name'] = df['Applicant Name'].str.strip()
df['Applicant Name']

0            Wanda Doe
1          Peter Jones
2       Wanda Maximoff
3         Peter Parker
4         Bruce Rogers
             ...      
1045    Diana Maximoff
1046       Steve Allen
1047    Charlie Rogers
1048       Peter Wayne
1049     Charlie Brown
Name: Applicant Name, Length: 1050, dtype: object

In [32]:
df['Email'] = df['Email'].str.split('@').str[1]
df['Email']

0            gmail.com
1              edu.org
2            yahoo.com
3            yahoo.com
4            stark.com
             ...      
1045           edu.org
1046           edu.org
1047         stark.com
1048    dailybugle.com
1049         yahoo.com
Name: Email, Length: 1050, dtype: object